In [ ]:
################# Perform spatial probability–based tumor-stroma boundary identification and calculate the distance to the boundary
#################
#################
#################
#################
#################
#################

In [ ]:
#################(1). Set the random seed
import random
random.seed(42)

In [ ]:
#################(2). Define the function to get tumor-stroma boundary
import os
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree
from scipy.ndimage import gaussian_filter
from sklearn.neighbors import NearestNeighbors, KNeighborsClassifier
import matplotlib.colors as mcolors
import anndata as ad

def calculate_boundary_metadata(obs_df, spatial_coords, obs_col, cat_a, cat_b, n_neighbors=20, cat_focus='Malignant', 
                               smooth_sigma=2, min_vertex_count=150, max_dist_threshold=300, 
                               focus_bias=0.1, save_dir=None, sample_name="sample"):

    # Obtain the categories present in the current data 
    existing_cats = obs_df[obs_col].unique()
    relevant_cats = [c for c in [cat_a, cat_b] if c in existing_cats]
    
    # If there are fewer than 2 categories, it is unable to calculate the boundaries
    if len(relevant_cats) < 2:
        print(f"Sample {sample_name}: Less than two categories found. Returning 0 distance.")
        valid_lines = []
        dist_to_boundary = np.zeros(len(obs_df)) 
        return valid_lines, dist_to_boundary

        
    X = spatial_coords
    labels = obs_df[obs_col].values

    # Identify the boundary cells
    nbrs = NearestNeighbors(n_neighbors=n_neighbors).fit(X)
    _, indices = nbrs.kneighbors(X)
    boundary_mask = np.zeros(len(obs_df), dtype=bool)
    
    for i in range(len(obs_df)):
        current_label = labels[i]
        if current_label not in [cat_a, cat_b]: continue
        neighbor_labels = labels[indices[i]]
        if (current_label == cat_a and cat_b in neighbor_labels) or \
           (current_label == cat_b and cat_a in neighbor_labels):
            boundary_mask[i] = True
    
    # Train the KNN to predict categories
    y = (obs_df[obs_col] == cat_focus).astype(int).values
    clf = KNeighborsClassifier(n_neighbors=n_neighbors, weights='distance').fit(X, y)
    
    # Build a grid
    padding = 0.1
    x_min, x_max = X[:, 0].min(), X[:, 0].max()
    y_min, y_max = X[:, 1].min(), X[:, 1].max()
    x_pad, y_pad = (x_max - x_min) * padding, (y_max - y_min) * padding
    
    grid_res = 500
    xx, yy = np.meshgrid(np.linspace(x_min - x_pad, x_max + x_pad, grid_res),
                         np.linspace(y_min - y_pad, y_max + y_pad, grid_res))
    grid_points = np.c_[xx.ravel(), yy.ravel()]
    
    # Calculate the grid mask in effective range 
    nbrs_geo = NearestNeighbors(n_neighbors=1).fit(X)
    dists_geo, _ = nbrs_geo.kneighbors(grid_points)
    dists_geo = dists_geo.reshape(xx.shape)
    
    Z_raw = clf.predict_proba(grid_points)[:, 1].reshape(xx.shape)
    Z_biased = np.clip(Z_raw + focus_bias, 0, 1) 
    Z_smooth = gaussian_filter(Z_biased, sigma=smooth_sigma)
    Z_smooth[dists_geo > max_dist_threshold] = np.nan
    
    # Plots
    fig, ax = plt.subplots(figsize=(10, 8), dpi=100)
    colors = {cat_a: "#B53E2B", cat_b: "#D6E7A3"}
    for cat, color in colors.items():
        mask = (obs_df[obs_col] == cat)
        ax.scatter(X[mask, 0], X[mask, 1], c=color, s=4, alpha=0.7, label=cat)
    
    contours = ax.contour(xx, yy, Z_smooth, levels=[0.5], alpha=0) 
    valid_lines = []
    if len(contours.allsegs) > 0:
        for seg in contours.allsegs[0]:
            if len(seg) > min_vertex_count:
                ax.plot(seg[:, 0], seg[:, 1], color='black', linewidth=4, zorder=10)
                valid_lines.append(seg)

    ax.set_title(f"Boundary: {sample_name}")
    ax.set_aspect('equal')
    if ax.get_ylim()[0] < ax.get_ylim()[1]: ax.invert_yaxis()
    
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        plt.savefig(os.path.join(save_dir, f"boundary_{sample_name}.png"), bbox_inches='tight')
    plt.show()
    plt.close(fig)
    
    if len(valid_lines) > 0:
        fig_long, ax_long = plt.subplots(figsize=(10, 8), dpi=100)
        for cat, color in colors.items():
            mask = (obs_df[obs_col] == cat)
            ax_long.scatter(X[mask, 0], X[mask, 1], c=color, s=4, alpha=0.7, label=cat)
        longest_line = max(valid_lines, key=lambda seg: len(seg))
        ax_long.plot(longest_line[:, 0], longest_line[:, 1], color='black', linewidth=4, zorder=10)
        
        ax_long.set_title(f"Boundary Longest: {sample_name}")
        ax_long.set_aspect('equal')
        if ax_long.get_ylim()[0] < ax_long.get_ylim()[1]: ax_long.invert_yaxis()
        
        plt.savefig(os.path.join(save_dir, f"boundary_longest_{sample_name}.png"), bbox_inches='tight')
        plt.close(fig_long)
        

    # Calculate the distance to the boundary
    dist_to_boundary_all = np.full(len(obs_df), np.nan)
    if len(valid_lines) > 0:
        all_pts = np.vstack(valid_lines)
        tree_all = cKDTree(all_pts)
        dist_to_boundary_all, _ = tree_all.query(spatial_coords, k=1)
    
    # Calculate the distance to the longest boundary
    dist_to_boundary_longest = np.full(len(obs_df), np.nan)
    longest_line = None
    if len(valid_lines) > 0:
        longest_line = max(valid_lines, key=lambda seg: len(seg))
        tree_longest = cKDTree(longest_line)
        dist_to_boundary_longest, _ = tree_longest.query(spatial_coords, k=1)
    return valid_lines, dist_to_boundary_all, dist_to_boundary_longest, longest_line

In [ ]:
#################(3). Define the function to perform tumor-stroma region segmentation
from shapely.geometry import Point, Polygon, LineString
from shapely.ops import split
import alphashape

def calculate_geometric_zoning_metadata(obs_df, spatial_coords, boundary_coords, obs_col, 
                          cat_focus='Malignant', save_dir=None, sample_name="sample"):
    results = {}  
    def get_zoning_result(use_longest_only):
        X = spatial_coords
        tissue_poly = alphashape.alphashape(X, 0.01)
        if not isinstance(tissue_poly, Polygon) and hasattr(tissue_poly, 'geoms'):
            tissue_poly = max(tissue_poly.geoms, key=lambda p: p.area)

        # Get boundary lines
        all_lines = []
        for coords in boundary_coords:
            if len(coords) < 10: continue
            p1, p2, pn_1, pn = coords[0], coords[1], coords[-2], coords[-1]
            ext_start = p1 + (p1 - p2) * 0.2
            ext_end = pn + (pn - pn_1) * 0.2
            all_lines.append(LineString(np.vstack([ext_start, coords, ext_end])))
        
        target_lines = [max(all_lines, key=lambda l: l.length)] if use_longest_only and all_lines else all_lines
        
        # Region segmentation
        regions = [tissue_poly]
        for line in target_lines:
            new_regions = []
            for reg in regions:
                parts = split(reg, line)
                new_regions.extend(list(parts.geoms) if len(parts.geoms) > 1 else [reg])
            regions = new_regions
            
        temp_zone = pd.Series("None", index=obs_df.index)
        for i, poly in enumerate(regions):
            is_inside = [poly.covers(Point(p)) for p in X]
            temp_zone[is_inside] = f"Zone_{i}"
            
        stats = obs_df.groupby(temp_zone)[obs_col].value_counts(normalize=True).unstack().fillna(0)
        Epithelial_Zones = stats.index[stats[cat_focus] >= 0.5].tolist() if cat_focus in stats.columns else []
        spatial_zone = np.where(temp_zone.isin(Epithelial_Zones), 'Epithelial_Zone', 'Other_Zone')
        
        return pd.Series(spatial_zone, index=obs_df.index), regions, target_lines, stats

    # Plot region segmentation results
    for mode in ['all', 'longest']:
        use_long = (mode == 'longest')
        zone_series, regions, lines, stats = get_zoning_result(use_long)
        results[mode] = zone_series
        
        fig, ax = plt.subplots(figsize=(10, 10), dpi=100)
        colors = {"Epithelial_Zone": "#B53E2B", "Other_Zone": "#D6E7A3"}
        for z_type in ["Epithelial_Zone", "Other_Zone"]:
            mask = (zone_series == z_type)
            ax.scatter(spatial_coords[mask, 0], spatial_coords[mask, 1], c=colors[z_type], s=3, alpha=0.6)
            
        for i, poly in enumerate(regions):
            rep = poly.representative_point()
            ratio = stats.loc[f"Zone_{i}", cat_focus] if f"Zone_{i}" in stats.index and cat_focus in stats.columns else 0
            ax.text(rep.x, rep.y, f"Z{i}\n({ratio:.1%})", fontsize=8, fontweight='bold', ha='center', bbox=dict(fc="white", alpha=0.7))

        if save_dir:
            os.makedirs(save_dir, exist_ok=True)
            plt.savefig(os.path.join(save_dir, f"zoning_{mode}_{sample_name}.png"))
        plt.close()

    return results['all'], results['longest']

In [ ]:
#################(4). Load meta data and set the cell type to Malignant and non-Malignant
space_meta=pd.read_csv('/data/wangjiahao/space/seurat_niche/seurat_niches.csv',index_col=0)
space_meta['binary_type'] = np.where(
    space_meta['celltype'] == 'Malignant', 
    'Malignant', 
    'Other'
)

In [ ]:
#################(5). Calculate the tumor-stroma boundary and distance to the boundary for each cancer type
import pickle
# Get all unique cancer types present in the metadata
cancer_types = space_meta['cancertype'].unique()

all_boundary_coords = {}
all_boundary_coords_longest = {}
all_results_list = []

# Loop through each cancer type
for cancer_type in cancer_types:
    # Set the output directory based on the current cancer type
    output_base_dir = f"boundary_analysis_results_{cancer_type}"
    os.makedirs(output_base_dir, exist_ok=True)
    
    # Get all unique samples belonging to the current cancer type
    samples = space_meta[space_meta['cancertype'] == cancer_type]['sample'].unique()
    
    for sample_id in samples:
        print(f"\n>>> Processing sample: {sample_id} ({cancer_type})")
        
        # 1. Filter rows for the current sample
        mask = space_meta['sample'] == sample_id
        sample_df = space_meta[mask].copy()
        
        # 2. Extract spatial coordinate matrix (X, Y)
        spatial_coords = sample_df[['coord_x', 'coord_y']].values
        
        try:
            # Step 1: Calculate the boundary line
            # Returns: boundary_lines (list of segments), dists (numpy array)
            boundary_lines, dist_all, dist_longest, longest_line = calculate_boundary_metadata(
                obs_df=sample_df,
                spatial_coords=spatial_coords,
                obs_col='binary_type',
                cat_a='Malignant',
                cat_b='Other',
                n_neighbors=200, 
                min_vertex_count=100,
                cat_focus='Malignant',
                max_dist_threshold=200,
                save_dir=output_base_dir,
                sample_name=sample_id,
                focus_bias=0.2
            )
            all_boundary_coords[sample_id] = boundary_lines
            all_boundary_coords_longest[sample_id] = longest_line
            
            # Step 2: Geometric zoning (Zone 0, Zone 1 -> Tumor_Zone, Normal_Zone)
            # Returns: zoning_array (numpy array)
            zoning_all, zoning_longest = calculate_geometric_zoning_metadata(
                obs_df=sample_df,
                spatial_coords=spatial_coords,
                boundary_coords=boundary_lines,
                obs_col="binary_type",
                cat_focus='Malignant',
                save_dir=output_base_dir,
                sample_name=sample_id
            )
            
            # 3. Integrate results for the current sample
            res = pd.DataFrame({
                'cellid': sample_df['Cell_ID'].values, # Use the cell_id column from metadata
                'zoning_all': zoning_all.values,
                'zoning_longest': zoning_longest.values,
                'dist_to_boundary_all': dist_all,
                'dist_to_boundary_longest': dist_longest,
                'sample': sample_id,
                'cancertype': cancer_type
            }, index=sample_df.index)
            
            all_results_list.append(res)
            print(f"[{sample_id}] Processing completed.")
            
        except Exception as e:
            print(f"[{sample_id}] Error occurred during runtime: {e}")

    # 4. Merge results and save for the current cancer type
    type_results = [r for r in all_results_list if r['cancertype'].iloc[0] == cancer_type]
    
    if type_results:
        final_zoning_df = pd.concat(type_results)
        final_zoning_df.to_csv(f"{output_base_dir}/final_boundary_with_zones.csv")
        print(f"\n[{cancer_type}] Results merged and exported to {output_base_dir}.")

    # Save the boundary lines using pickle for the current cancer type
    with open(f"{output_base_dir}/all_samples_boundary_lines.pkl", "wb") as f:
        pickle.dump({k: v for k, v in all_boundary_coords.items() if k in samples}, f)
        
    with open(f"{output_base_dir}/all_samples_boundary_longestlines.pkl", "wb") as f:
        pickle.dump({k: v for k, v in all_boundary_coords_longest.items() if k in samples}, f)

print("\nAll cancer types and tasks are completely finished!")